## Training Time Scaling vs Inference Time Scaling

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

API_KEY = os.getenv("GROQ_API_KEY")
BASE_URL = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=API_KEY, base_url=BASE_URL)


In [3]:
# Check the linked models

models = groq.models.list()
for model in models.data:
    display(Markdown(f"**Model ID:** {model.id}"))
    display(Markdown(f"**Object Type:** {model.object}"))
    display(Markdown(f"**Created At:** {model.created}"))
    display(Markdown(f"**Owned By:** {model.owned_by}"))
    display(Markdown("---"))

**Model ID:** whisper-large-v3

**Object Type:** model

**Created At:** 1693721698

**Owned By:** OpenAI

---

**Model ID:** meta-llama/llama-prompt-guard-2-86m

... (45 outputs left out)

**Object Type:** model

**Created At:** 1754408224

**Owned By:** OpenAI

---

In [4]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [ ]:
response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=easy_puzzle, reasoning_effort="medium")
display(Markdown(response.choices[0].message.content))

# Try with Different models with different reasoning efforts and see how the answer changes.
# With different reasoning efforts, the model may provide different answers to the same question. You can experiment with different models and reasoning efforts to see how they affect the output.

\( \frac{2}{3} \)

In [ ]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [ ]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]

## Use Absatction Layers (Langchain)

### HEavy Weight Langchain

In [ ]:
from langchain_openai import ChatOpenAI

tell_a_joke = [
    {"role": "user", "content": "Tell me a joke."}
]

llm = ChatOpenAI(model="gpt-6-luna", reasoning_effort="none")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

c:\Users\saree\Desktop\AI ENG\AI-Engineering\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Why did the scarecrow win an award?

Because he was outstanding in his field.

### Lightweight Light LLMS

In [12]:
from litellm import completion
import litellm

# Prints the list of provider model strings hardcoded/supported natively
print(litellm.models)

<module 'litellm.models' from 'c:\\Users\\saree\\Desktop\\AI ENG\\AI-Engineering\\.venv\\Lib\\site-packages\\litellm\\models\\__init__.py'>


In [27]:
from litellm import completion
import os
os.environ["GROQ_API_KEY"] = API_KEY
response = completion(model="groq/openai/gpt-oss-120b", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why don’t scientists trust atoms anymore?

Because they *make up* everything! 😄

In [28]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 76
Output tokens: 46
Total tokens: 122
Total cost: 0.0039 cents
